# Overview
## Training Model

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [3]:
df = pd.read_csv("Student_data.csv")
df.head()
df = df.drop(columns="Unnamed: 0", axis=1)

In [4]:
X = df.drop(columns=['math score', 'average_score', 'total_score'])
X.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [5]:
Y = df["math score"]
Y.head()

0    72
1    69
2    90
3    47
4    76
Name: math score, dtype: int64

In [6]:
num_features = X.select_dtypes(exclude=object).columns
cat_features = X.select_dtypes(include=object).columns

In [7]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_tranformer = StandardScaler()
oh_tranformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_tranformer, cat_features), 
        ("StandardScaler", numeric_tranformer, num_features)
    ]
)

In [8]:
X = preprocessor.fit_transform(X)
X.shape


(1000, 18)

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train , Y_test = train_test_split(X,Y, test_size=0.20, random_state=42)

In [12]:
X_train.shape, X_test.shape

((800, 18), (200, 18))

In [13]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [18]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]
for name, model in models.items():
    
    model.fit(X_train, Y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(Y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(Y_test, y_test_pred)

    
    print(name)
    model_list.append(model)
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

/Users/palak/Desktop/Project/ML project (Krish)/myenv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/palak/Desktop/Project/ML project (Krish)/myenv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/palak/Desktop/Project/ML project (Krish)/myenv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/palak/Desktop/Project/ML project (Krish)/myenv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/palak/Desktop/Project/ML project (Krish)/myenv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 5.3233
- Mean Absolute Error: 4.2687
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3918
- Mean Absolute Error: 4.2114
- R2 Score: 0.8805


Lasso
Model performance for Training set
- Root Mean Squared Error: 6.5938
- Mean Absolute Error: 5.2063
- R2 Score: 0.8071
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 6.5197
- Mean Absolute Error: 5.1579
- R2 Score: 0.8253


Ridge
Model performance for Training set
- Root Mean Squared Error: 5.3236
- Mean Absolute Error: 4.2669
- R2 Score: 0.8743
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 5.3881
- Mean Absolute Error: 4.2076
- R2 Score: 0.8807


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 5.6998
- Mean Absolute Error: 4.5237
- R2 Score: 0.8559
-----------------------

In [19]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
2,Ridge(),0.880694
0,LinearRegression(),0.880533
5,"(DecisionTreeRegressor(max_features=1.0, rando...",0.853240
7,"CatBoostRegressor(loss_function='RMSE', verbos...",0.848197
8,"(DecisionTreeRegressor(max_depth=3, random_sta...",0.845639
1,Lasso(),0.825320
6,"XGBRegressor(base_score=None, booster=None, ca...",0.819980
3,KNeighborsRegressor(),0.779470
4,DecisionTreeRegressor(),0.754396


In [20]:
! git add .

In [21]:
! git commit -m "Model Training in jy-notebook"

[main 3700b0f] Model Training in jy-notebook
 7 files changed, 4729 insertions(+), 34 deletions(-)
 create mode 100644 notebook/Model_training.ipynb
 create mode 100644 notebook/Student_data.csv
 create mode 100644 notebook/catboost_info/catboost_training.json
 create mode 100644 notebook/catboost_info/learn/events.out.tfevents
 create mode 100644 notebook/catboost_info/learn_error.tsv
 create mode 100644 notebook/catboost_info/time_left.tsv


In [22]:
! git push -u origin main

Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 12 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (12/12), 60.38 KiB | 15.09 MiB/s, done.
Total 12 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/palaknagar-07/ML-project-End-to-End-.git
   f5aa2a0..3700b0f  main -> main
branch 'main' set up to track 'origin/main'.
